In [1]:
# ════════════════════════════════════════════════════════════
# CELL 1 — Install + patch import bug
# ════════════════════════════════════════════════════════════
import subprocess, sys
subprocess.check_call([sys.executable, "-m", "pip", "-q", "install",
                       "realesrgan", "basicsr", "facexlib", "gfpgan"])

# Patch broken torchvision import BEFORE importing basicsr
import types, torchvision.transforms.functional as _ftv, sys as _sys
_mock = types.ModuleType("torchvision.transforms.functional_tensor")
_mock.rgb_to_grayscale = _ftv.rgb_to_grayscale
_sys.modules["torchvision.transforms.functional_tensor"] = _mock

from basicsr.archs.rrdbnet_arch import RRDBNet
print("✅ basicsr imported successfully")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 172.5/172.5 kB 5.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.8/46.8 kB 3.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 178.0/178.0 kB 13.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.6/59.6 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.2/52.2 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 333.1/333.1 kB 21.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 94.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 256.2/256.2 kB 19.4 MB/s eta 0:00:00
✅ basicsr imported successfully


In [15]:
# ════════════════════════════════════════════════════════════
# CELL 2 — Config
# ════════════════════════════════════════════════════════════
import os, math, random, copy, time
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from PIL import Image
from pathlib import Path
import torchvision.transforms.functional as TF
from torch.utils.data import Dataset, DataLoader
import pandas as pd
from tqdm import tqdm

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"GPUs: {torch.cuda.device_count()}")

CFG = dict(
    TRAIN_LR   = "/kaggle/input/competitions/dlp-jan-2026-nppe-3/train/train/input",
    TRAIN_HR   = "/kaggle/input/competitions/dlp-jan-2026-nppe-3/train/train/ground_truth",
    TEST_LR    = "/kaggle/input/competitions/dlp-jan-2026-nppe-3/test/test/input",
    SAMPLE_CSV = "/kaggle/input/competitions/dlp-jan-2026-nppe-3/sample_submission.csv",
    OUTPUT_DIR = "/kaggle/working/predictions",
    CKPT_DIR   = "/kaggle/working/checkpoints",

    # Real-ESRGAN exact architecture
    N_RRDB     = 23,       # must match pretrained weights exactly
    SCALE      = 4,

    # Fine-tuning config
    PATCH_SIZE = 64,
    BATCH_SIZE = 16,
    EPOCHS     = 40,       
    LR         = 2e-4,     # LOW LR — preserving pretrained knowledge
    LR_MIN     = 1e-7,
    EMA_DECAY  = 0.999,    # slow EMA — weights already good
    NUM_WORKERS= 4,

    GAMMA_RANGE  = (0.6, 1.8),
    BRIGHT_RANGE = (0.8, 1.2),

    TILE_SIZE    = 128,
    TILE_OVERLAP = 16,
    USE_TTA      = True,
    SEED         = 42,
)

os.makedirs(CFG["OUTPUT_DIR"], exist_ok=True)
os.makedirs(CFG["CKPT_DIR"],   exist_ok=True)

def seed_everything(s):
    random.seed(s); np.random.seed(s)
    torch.manual_seed(s); torch.cuda.manual_seed_all(s)
    torch.backends.cudnn.benchmark = True

seed_everything(CFG["SEED"])

GPU: Tesla T4
GPUs: 2


In [3]:
# ════════════════════════════════════════════════════════════
# CELL 3 — Download pretrained weights
# ════════════════════════════════════════════════════════════
import urllib.request

weight_path = "/kaggle/working/RealESRGAN_x4plus.pth"
if not os.path.exists(weight_path):
    print("Downloading pretrained weights...")
    urllib.request.urlretrieve(
        "https://github.com/xinntao/Real-ESRGAN/releases/download/v0.1.0/RealESRGAN_x4plus.pth",
        weight_path
    )
print(f"✅ Weights: {os.path.getsize(weight_path)/1e6:.1f}MB")

✅ Weights: 67.0MB


In [4]:
# ════════════════════════════════════════════════════════════
# CELL 4 — Build model + load pretrained
# ════════════════════════════════════════════════════════════

def build_model():
    """Real-ESRGAN RRDBNet with exact pretrained architecture."""
    model = RRDBNet(
        num_in_ch=3, num_out_ch=3,
        num_feat=64, num_block=23,
        num_grow_ch=32, scale=4
    ).to(DEVICE)
    return model


def load_pretrained(model):
    """Load Real-ESRGAN pretrained weights."""
    ckpt  = torch.load(weight_path, map_location=DEVICE)
    # Real-ESRGAN uses 'params_ema' key
    state = ckpt.get("params_ema", ckpt.get("params", ckpt))

    # Strip any prefix
    clean = {}
    for k, v in state.items():
        nk = k
        if nk.startswith("module."): nk = nk[7:]
        clean[nk] = v

    missing, unexpected = model.load_state_dict(clean, strict=False)
    print(f"✅ Pretrained loaded | missing={len(missing)} unexpected={len(unexpected)}")
    return model


# Test
_m = build_model()
_m = load_pretrained(_m)
with torch.no_grad():
    _y = _m(torch.zeros(1,3,64,64,device=DEVICE))
print(f"Shape test: (1,3,64,64) → {tuple(_y.shape)}")
del _m, _y

✅ Pretrained loaded | missing=0 unexpected=0
Shape test: (1,3,64,64) → (1, 3, 256, 256)


In [5]:
# ════════════════════════════════════════════════════════════
# CELL 5 — Dataset (same as before, no CLAHE)
# ════════════════════════════════════════════════════════════

IMG_EXTS = {".jpg",".jpeg",".png",".bmp"}

def auto_brighten(img_np: np.ndarray) -> np.ndarray:
    """Percentile stretch to normalize dark images to visible range."""
    img_f  = img_np.astype(np.float32)
    p_low  = np.percentile(img_f, 2)
    p_high = np.percentile(img_f, 98)
    if p_high - p_low < 10:
        p_low  = img_f.min()
        p_high = np.percentile(img_f, 90)
    if p_high <= p_low:
        return img_np
    out = (img_f - p_low) / (p_high - p_low) * 255.0
    return np.clip(out, 0, 255).astype(np.uint8)



class SRDataset(Dataset):
    def __init__(self, lr_dir, hr_dir, patch_size=64, augment=True):
        self.patch_size = patch_size
        self.augment    = augment
        self.lr_paths = sorted(p for p in Path(lr_dir).iterdir()
                               if p.suffix.lower() in IMG_EXTS)
        self.hr_paths = sorted(p for p in Path(hr_dir).iterdir()
                               if p.suffix.lower() in IMG_EXTS)
        assert len(self.lr_paths) == len(self.hr_paths)
        lr0 = Image.open(self.lr_paths[0])
        hr0 = Image.open(self.hr_paths[0])
        self.sh = hr0.size[1] / lr0.size[1]
        self.sw = hr0.size[0] / lr0.size[0]
        print(f"Dataset: {len(self.lr_paths)} pairs | "
              f"Scale {self.sw:.4f}×W {self.sh:.4f}×H")

    def __len__(self): return len(self.lr_paths)

    def _paired_crop(self, lr, hr):
        _, lH, lW = lr.shape
        _, hH, hW = hr.shape
        ps = self.patch_size
        if lH < ps or lW < ps:
            ph=max(0,ps-lH); pw=max(0,ps-lW)
            lr=TF.pad(lr,[0,0,pw,ph],padding_mode="reflect")
            hr=TF.pad(hr,[0,0,pw*4,ph*4],padding_mode="reflect")
            _,lH,lW=lr.shape; _,hH,hW=hr.shape
        top  = random.randint(0, lH-ps)
        left = random.randint(0, lW-ps)
        ht = min(round(top*(hH/lH)),  hH-ps*4)
        hl = min(round(left*(hW/lW)), hW-ps*4)
        lr_c = lr[:, top:top+ps, left:left+ps]
        hr_c = hr[:, ht:ht+ps*4, hl:hl+ps*4]
        if hr_c.shape[-2]!=ps*4 or hr_c.shape[-1]!=ps*4:
            hr_c = TF.resize(hr_c,[ps*4,ps*4],
                             interpolation=TF.InterpolationMode.BICUBIC,
                             antialias=True).clamp(0,1)
        return lr_c, hr_c

    def _aug(self, lr, hr):
        if random.random()>0.5: lr,hr=TF.hflip(lr),TF.hflip(hr)
        if random.random()>0.5: lr,hr=TF.vflip(lr),TF.vflip(hr)
        k=random.randint(0,3)
        if k: lr=torch.rot90(lr,k,[-2,-1]); hr=torch.rot90(hr,k,[-2,-1])
        g=random.uniform(*CFG["GAMMA_RANGE"])
        b=random.uniform(*CFG["BRIGHT_RANGE"])
        lr=(lr**g*b).clamp(0,1); hr=(hr**g*b).clamp(0,1)
        return lr, hr

    def __getitem__(self, idx):
        # Brighten dark LR so pretrained model recognizes it
        lr_np  = np.array(Image.open(self.lr_paths[idx]).convert("RGB"))
        lr_np  = auto_brighten(lr_np)                    # ← ADD THIS
        lr     = TF.to_tensor(Image.fromarray(lr_np))
        hr     = TF.to_tensor(Image.open(self.hr_paths[idx]).convert("RGB"))
        if self.augment:
            lr, hr = self._paired_crop(lr, hr)
            lr, hr = self._aug(lr, hr)
        else:
            lr, hr = self._paired_crop(lr, hr)
        return lr, hr

In [6]:
# ════════════════════════════════════════════════════════════
# CELL 6 — Loss + EMA + metrics
# ════════════════════════════════════════════════════════════

class CharbonnierLoss(nn.Module):
    def __init__(self, eps=1e-3):
        super().__init__()
        self.e2 = eps**2
    def forward(self, p, t):
        return torch.sqrt((p-t)**2 + self.e2).mean()


class ColorLoss(nn.Module):
    def forward(self, p, t):
        return (p.mean(dim=[2,3]) - t.mean(dim=[2,3])).abs().mean()


class FineTuneLoss(nn.Module):
    def __init__(self):
        super().__init__()
        self.char  = CharbonnierLoss()
        self.color = ColorLoss()
    def forward(self, pred, target):
        lc = self.char(pred, target)
        ll = self.color(pred, target)
        return lc + 0.1*ll, lc.item(), ll.item()


class EMA:
    def __init__(self, model, decay=0.999):
        self.decay  = decay
        self.shadow = copy.deepcopy(model).eval()
        for p in self.shadow.parameters():
            p.requires_grad_(False)

    @torch.no_grad()
    def update(self, model):
        for s,m in zip(self.shadow.parameters(),
                       model.parameters()):
            s.copy_(s*self.decay + m*(1-self.decay))


def kaggle_rmse(sr_t, hr_t):
    """Exact Kaggle metric — grayscale linspace sampling."""
    if sr_t.shape[-2:] != hr_t.shape[-2:]:
        sr_t = F.interpolate(sr_t.unsqueeze(0),
                             size=hr_t.shape[-2:],
                             mode="bilinear",
                             align_corners=False).squeeze(0)
    sg = (0.299*sr_t[0]+0.587*sr_t[1]+0.114*sr_t[2]).clamp(0,1)*255
    hg = (0.299*hr_t[0]+0.587*hr_t[1]+0.114*hr_t[2]).clamp(0,1)*255
    idx = torch.linspace(0, sg.numel()-1, 100, dtype=torch.long)
    return ((sg.flatten()[idx]-hg.flatten()[idx])**2).mean().sqrt().item()

In [10]:
# ════════════════════════════════════════════════════════════
# CELL 7 — Full image validation
# ════════════════════════════════════════════════════════════

@torch.no_grad()
def full_validate(model, n=50):
    model.eval()
    lr_paths = sorted(Path(CFG["TRAIN_LR"]).iterdir())[:n]
    hr_paths = sorted(Path(CFG["TRAIN_HR"]).iterdir())[:n]
    rmses    = []

    for lp, hp in zip(lr_paths, hr_paths):
        lr_np = np.array(Image.open(lp).convert("RGB"))
        lr_np = auto_brighten(lr_np)  
        lr_t  = TF.to_tensor(Image.fromarray(lr_np))
        hr_t = TF.to_tensor(Image.open(hp).convert("RGB"))
        C,H,W = lr_t.shape
        ts,ov = CFG["TILE_SIZE"], CFG["TILE_OVERLAP"]
        step  = max(1, ts-ov)
        oH,oW = H*4, W*4
        out   = torch.zeros(1,C,oH,oW,device=DEVICE)
        wt    = torch.zeros(1,1,oH,oW,device=DEVICE)
        lr_b  = lr_t.unsqueeze(0).to(DEVICE)

        ys = list(range(0,max(1,H-ts+1),step))
        xs = list(range(0,max(1,W-ts+1),step))
        if H<=ts: ys=[0]
        else: ys=list(set(ys+[H-ts]))
        if W<=ts: xs=[0]
        else: xs=list(set(xs+[W-ts]))

        for y in ys:
            for x in xs:
                y1=max(0,min(y,H-ts)) if H>ts else 0
                x1=max(0,min(x,W-ts)) if W>ts else 0
                y2=min(y1+ts,H); x2=min(x1+ts,W)
                with torch.amp.autocast('cuda'):
                    sr = model(lr_b[:,:,y1:y2,x1:x2])
                th,tw=(y2-y1)*4,(x2-x1)*4
                out[:,:,y1*4:y1*4+th,x1*4:x1*4+tw] += sr
                wt[ :,:,y1*4:y1*4+th,x1*4:x1*4+tw] += 1

        sr = (out/wt.clamp(1e-6)).squeeze(0).clamp(0,1).cpu()
        rmses.append(kaggle_rmse(sr, hr_t))

    mean_r = float(np.mean(rmses))
    print(f"  📊 True RMSE ({n} images): {mean_r:.3f} ± {np.std(rmses):.3f}")
    return mean_r

In [8]:
def load_pretrained(model):
    ckpt  = torch.load(weight_path, map_location=DEVICE)
    state = ckpt.get("params_ema", ckpt.get("params", ckpt))

    # Strip prefix
    clean = {(k[7:] if k.startswith("module.") else k):v
             for k,v in state.items()}

    missing, unexpected = model.load_state_dict(clean, strict=False)
    
    # ── Verify weights actually loaded ──────────────────
    total   = len(list(model.parameters()))
    matched = total - len(missing)
    print(f"Total keys     : {len(clean)}")
    print(f"Missing keys   : {len(missing)}")
    print(f"Unexpected keys: {len(unexpected)}")
    
    if len(missing) > 10:
        print("❌ WEIGHTS DID NOT LOAD — key mismatch")
        print("Missing sample:", missing[:3])
    else:
        print("✅ Weights loaded successfully")
    
    return model

In [16]:
def finetune():
    # ── Dataset ──────────────────────────────────────────
    ds = SRDataset(CFG["TRAIN_LR"], CFG["TRAIN_HR"],
                   CFG["PATCH_SIZE"], augment=True)
    loader = DataLoader(ds, batch_size=CFG["BATCH_SIZE"],
                        shuffle=True, num_workers=CFG["NUM_WORKERS"],
                        pin_memory=True, drop_last=True)
    print(f"Train: {len(ds)} images | {len(loader)} steps/epoch")

    # ── Model with pretrained weights ────────────────────
    model = build_model()
    model = load_pretrained(model)

    if torch.cuda.device_count() > 1:
        print(f"Using {torch.cuda.device_count()} GPUs")
        model = nn.DataParallel(model)

    raw    = model.module if hasattr(model, "module") else model
    ema    = EMA(raw, decay=CFG["EMA_DECAY"])
    crit   = FineTuneLoss().to(DEVICE)
    opt    = torch.optim.AdamW(model.parameters(),
                                lr=CFG["LR"],
                                weight_decay=1e-4,
                                betas=(0.9, 0.99))
    scaler = torch.amp.GradScaler('cuda')
    steps  = len(loader)

    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        opt, T_max=CFG["EPOCHS"] * steps, eta_min=CFG["LR_MIN"]
    )

    # ── Resume from checkpoint if exists ─────────────────
    start_epoch = 0
    best_rmse   = float("inf")
    ckpt_path   = f"{CFG['CKPT_DIR']}/last.pth"

    if os.path.exists(ckpt_path):
        ck    = torch.load(ckpt_path, map_location=DEVICE)
        strip = lambda sd: {(k[7:] if k.startswith("module.") else k):v
                            for k,v in sd.items()}
        raw.load_state_dict(strip(ck.get("ema", ck["model"])))
        try:
            opt.load_state_dict(ck["optimizer"])
        except:
            pass
        start_epoch = ck["epoch"]
        best_rmse   = ck["best_rmse"]
        # Fast-forward scheduler
        for _ in range(start_epoch * steps):
            scheduler.step()
        print(f"✅ Resumed from epoch {start_epoch} | best RMSE {best_rmse:.3f}")
    else:
        # Validate pretrained baseline only on fresh start
        print("\nPretrained baseline (no fine-tuning):")
        full_validate(raw, n=50)

    true_rmse_history = []

    for epoch in range(start_epoch, CFG["EPOCHS"]):
        model.train()
        ep_loss = 0.0
        t0      = time.time()
        opt.zero_grad(set_to_none=True)

        for step, (lr_imgs, hr_imgs) in enumerate(loader, 1):
            lr_imgs = lr_imgs.to(DEVICE, non_blocking=True)
            hr_imgs = hr_imgs.to(DEVICE, non_blocking=True)

            with torch.amp.autocast('cuda'):
                pred = model(lr_imgs)
                loss, lc, ll = crit(pred, hr_imgs)

            if torch.isnan(loss) or torch.isinf(loss):
                opt.zero_grad(set_to_none=True)
                continue

            scaler.scale(loss).backward()
            scaler.unscale_(opt)
            nn.utils.clip_grad_norm_(model.parameters(), 0.5)
            scaler.step(opt)
            scaler.update()
            opt.zero_grad(set_to_none=True)
            ema.update(raw)
            scheduler.step()
            ep_loss += loss.item()

        elapsed = time.time() - t0
        lr_now  = scheduler.get_last_lr()[0]
        print(f"[Ep {epoch+1:02d}/{CFG['EPOCHS']}] "
              f"Loss={ep_loss/steps:.4f} | "
              f"LR={lr_now:.2e} | {elapsed:.0f}s")

        # ── True RMSE every epoch ─────────────────────────
        true_rmse = full_validate(ema.shadow, n=50)
        true_rmse_history.append(true_rmse)

        # ── Save every epoch with epoch number ───────────
        ckpt = {
            "epoch":     epoch + 1,
            "model":     raw.state_dict(),
            "ema":       ema.shadow.state_dict(),
            "optimizer": opt.state_dict(),
            "best_rmse": best_rmse,
        }
        torch.save(ckpt, f"{CFG['CKPT_DIR']}/last.pth")
        torch.save(ckpt, f"{CFG['CKPT_DIR']}/epoch_{epoch+1:03d}.pth")
        print(f"  💾 Saved epoch_{epoch+1:03d}.pth")

        if true_rmse < best_rmse:
            best_rmse = true_rmse
            ckpt["best_rmse"] = best_rmse
            torch.save(ckpt, f"{CFG['CKPT_DIR']}/best.pth")
            print(f"  ✅ New best: {best_rmse:.3f} → best.pth saved")

        # ── LR restart at epoch 20 ── INSIDE loop ────────
        if epoch + 1 == 20:
            print("\n🔄 LR Restart at epoch 20!")
            for pg in opt.param_groups:
                pg["lr"] = 1e-4
            scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
                opt,
                T_max   = 20 * steps,
                eta_min = CFG["LR_MIN"]
            )

        # ── Early stop if True RMSE rises 3 times ────────
        if len(true_rmse_history) >= 3:
            if all(true_rmse_history[-i] > true_rmse_history[-i-1]
                   for i in range(1, 3)):
                print(f"\n⛔ Early stop — True RMSE rising for 2 epochs")
                break

    print(f"\nBest True RMSE: {best_rmse:.3f}")
    return best_rmse


best = finetune()

Dataset: 1200 pairs | Scale 4.0064×W 4.0064×H
Train: 1200 images | 75 steps/epoch
Total keys     : 702
Missing keys   : 0
Unexpected keys: 0
✅ Weights loaded successfully
Using 2 GPUs
✅ Resumed from epoch 14 | best RMSE 31.189


/tmp/ipykernel_57/3846574305.py:50: UserWarning: Detected call of `lr_scheduler.step()` before `optimizer.step()`. In PyTorch 1.1.0 and later, you should call them in the opposite order: `optimizer.step()` before `lr_scheduler.step()`.  Failure to do this will result in PyTorch skipping the first value of the learning rate schedule. See more details at https://pytorch.org/docs/stable/optim.html#how-to-adjust-learning-rate
  scheduler.step()


[Ep 15/40] Loss=0.1051 | LR=2.86e-05 | 87s
  📊 True RMSE (50 images): 44.299 ± 17.001
  💾 Saved epoch_015.pth
[Ep 16/40] Loss=0.1047 | LR=2.71e-05 | 88s
  📊 True RMSE (50 images): 42.761 ± 16.875
  💾 Saved epoch_016.pth
[Ep 17/40] Loss=0.1002 | LR=2.55e-05 | 87s
  📊 True RMSE (50 images): 41.317 ± 16.756
  💾 Saved epoch_017.pth
[Ep 18/40] Loss=0.1015 | LR=2.39e-05 | 88s
  📊 True RMSE (50 images): 39.985 ± 16.616
  💾 Saved epoch_018.pth
[Ep 19/40] Loss=0.1012 | LR=2.23e-05 | 88s
  📊 True RMSE (50 images): 38.743 ± 16.461
  💾 Saved epoch_019.pth
[Ep 20/40] Loss=0.0965 | LR=2.07e-05 | 88s
  📊 True RMSE (50 images): 37.559 ± 16.310
  💾 Saved epoch_020.pth

🔄 LR Restart at epoch 20!
[Ep 21/40] Loss=0.1027 | LR=9.94e-05 | 87s
  📊 True RMSE (50 images): 36.551 ± 16.138
  💾 Saved epoch_021.pth
[Ep 22/40] Loss=0.1035 | LR=9.76e-05 | 87s
  📊 True RMSE (50 images): 35.587 ± 16.006
  💾 Saved epoch_022.pth
[Ep 23/40] Loss=0.1021 | LR=9.46e-05 | 87s
  📊 True RMSE (50 images): 34.684 ± 15.779
  💾 Sav

In [17]:
# ════════════════════════════════════════════════════════════
# CELL 9 — Inference
# ════════════════════════════════════════════════════════════

@torch.no_grad()
def tile_inference(model, lr_t, tile_size, overlap):
    C,H,W = lr_t.shape
    oH,oW = H*4,W*4
    out   = torch.zeros(1,C,oH,oW,device=DEVICE)
    wt    = torch.zeros(1,1,oH,oW,device=DEVICE)
    step  = max(1,tile_size-overlap)
    lr_b  = lr_t.unsqueeze(0).to(DEVICE)

    def hann(h,w):
        wy=torch.hann_window(h,periodic=False,device=DEVICE)
        wx=torch.hann_window(w,periodic=False,device=DEVICE)
        return (wy.unsqueeze(1)*wx.unsqueeze(0)).view(1,1,h,w)

    ys=list(range(0,max(1,H-tile_size+1),step))
    xs=list(range(0,max(1,W-tile_size+1),step))
    if H<=tile_size: ys=[0]
    else: ys=list(set(ys+[H-tile_size]))
    if W<=tile_size: xs=[0]
    else: xs=list(set(xs+[W-tile_size]))

    for y in ys:
        for x in xs:
            y1=max(0,min(y,H-tile_size)) if H>tile_size else 0
            x1=max(0,min(x,W-tile_size)) if W>tile_size else 0
            y2=min(y1+tile_size,H); x2=min(x1+tile_size,W)
            with torch.amp.autocast('cuda'):
                sr=model(lr_b[:,:,y1:y2,x1:x2])
            th,tw=(y2-y1)*4,(x2-x1)*4
            mask=hann(th,tw)
            out[:,:,y1*4:y1*4+th,x1*4:x1*4+tw]+=sr*mask
            wt[ :,:,y1*4:y1*4+th,x1*4:x1*4+tw]+=mask

    return (out/wt.clamp(1e-6)).squeeze(0).clamp(0,1).cpu()


@torch.no_grad()
def tta_inference(model, lr_t, tile_size, overlap):
    preds=[]
    for aug in range(8):
        x=lr_t.clone()
        if aug%2==1: x=torch.flip(x,[-1])
        k=aug//2
        if k: x=torch.rot90(x,k,[-2,-1])
        p=tile_inference(model,x,tile_size,overlap)
        if k: p=torch.rot90(p,-k,[-2,-1])
        if aug%2==1: p=torch.flip(p,[-1])
        preds.append(p)
    return torch.stack(preds).mean(0)


def run_inference():
    model = build_model()
    ckpt  = torch.load(f"{CFG['CKPT_DIR']}/best.pth", map_location=DEVICE)
    strip = lambda sd: {(k[7:] if k.startswith("module.") else k):v
                        for k,v in sd.items()}
    model.load_state_dict(strip(ckpt.get("ema", ckpt["model"])))
    model.eval()
    if torch.cuda.device_count()>1:
        model=nn.DataParallel(model)

    out_dir = Path(CFG["OUTPUT_DIR"])
    out_dir.mkdir(exist_ok=True)
    paths   = sorted(p for p in Path(CFG["TEST_LR"]).iterdir()
                     if p.suffix.lower() in IMG_EXTS)
    print(f"Inferring {len(paths)} images...")

    for path in tqdm(paths):
        lr_np = np.array(Image.open(path).convert("RGB"))
        lr_np = auto_brighten(lr_np)
        lr_t  = TF.to_tensor(Image.fromarray(lr_np))
        sr   = tta_inference(model, lr_t,
                             CFG["TILE_SIZE"], CFG["TILE_OVERLAP"])

        C,H,W = lr_t.shape
        tH = round(H*(1250/312))
        tW = round(W*(1250/312))
        if sr.shape[-2]!=tH or sr.shape[-1]!=tW:
            sr = F.interpolate(sr.unsqueeze(0),size=(tH,tW),
                               mode="bilinear",
                               align_corners=False).squeeze(0).clamp(0,1)

        sr_np=(sr.permute(1,2,0).numpy()*255).clip(0,255).astype(np.uint8)
        Image.fromarray(sr_np).save(
            out_dir/f"{path.stem}.jpg", quality=95, subsampling=0)

    print(f"✅ Done → {out_dir}")


run_inference()

Inferring 300 images...


100%|██████████| 300/300 [34:58<00:00,  7.00s/it]

✅ Done → /kaggle/working/predictions


In [18]:
# ════════════════════════════════════════════════════════════
# CELL 10 — Generate submission
# ════════════════════════════════════════════════════════════

def generate_submission(
    folder  = CFG["OUTPUT_DIR"],
    sample  = CFG["SAMPLE_CSV"],
    out_csv = "/kaggle/working/submission.csv",
):
    df_s = pd.read_csv(sample)
    first = os.path.join(folder, df_s["ID"].iloc[0]+".jpg")
    if not os.path.exists(first):
        raise FileNotFoundError(f"Missing: {first}")

    flat = np.array(Image.open(first).convert("L")).size
    idx  = np.linspace(0, flat-1, 100, dtype=int)
    rows = []

    for img_id in df_s["ID"]:
        p   = os.path.join(folder, img_id+".jpg")
        arr = np.array(Image.open(p).convert("L")).flatten()
        if arr.size != flat:
            arr = np.array(
                Image.open(p).convert("L").resize((1250,1250),Image.BICUBIC)
            ).flatten()
            idx = np.linspace(0, arr.size-1, 100, dtype=int)
        rows.append([img_id, *arr[idx]])

    cols = ["ID"]+[f"pixel_{i}" for i in range(100)]
    out  = pd.DataFrame(rows, columns=cols)
    out.to_csv(out_csv, index=False)
    print(f"✅ submission.csv  shape={out.shape}")
    print(f"   Pixel range: [{out.iloc[:,1:].min().min():.0f}, "
          f"{out.iloc[:,1:].max().max():.0f}]")
    print(f"   Mean pixel : {out.iloc[:,1:].mean().mean():.1f}")
    return out


sub = generate_submission()

✅ submission.csv  shape=(300, 101)
   Pixel range: [0, 255]
   Mean pixel : 91.9
